# Imports and Settings

In [25]:
import sys
'''
Добавляем в список путей поиска модулей папку src,
которая находится на уровень выше.
Это нужно, чтобы импортировать модули из этой папки.
'''
sys.path.append('../src')  
import logging
import pickle
from functools import partial

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

from config import (
    SAMPLE_FRAC,
    PIPELINE_PATH,
    PARAMS_LIST,
    WEIGHTS_LIST,
    PRE_FEATURES,
    RAW_DATA_PATH,
    TEMP_DATA_PATH,
    TARGET_PATH,
    TRAIN_SIZE,
    SEED_SPLIT_DATASET,
    STRATIFY_COL,
    TEST_PREDICT_PATH,
    THRESHOLD,
    CAT_FEATURES,
    N_SPLITS,
    SEED,
    SHUFFLE,
    PROP_FEATURES_DICT,
    MEAN_FREQ_SOURCE_LIST,
    DROP_LIST,
    PARQUET_FILE_PATTERN,
    PREDICT_FILE_EXTENSION
)

from data_utils import (
    load_dataset,
    split_dataset_by_target,
    check_data_folder_and_count_files,
    make_file_path,
    save_predictions_with_id,
)

from preprocessing import (
    SampleMedianImputer,
    convert_all_to_numeric_pipeline,
    convert_all_to_int_pipeline,
    drop_duplicates_pipeline,
)

from feature_engineering import (
    rn_max_feature_pipeline,
    enc_paym_transcoding_pipeline,
    definite_value_proportion_features_pipeline,
    from_is_zero_prop_1_create_sum_prop_1_feature_pipeline,
    mean_value_frequency_feature_pipeline,
    enc_paym_norm_group_sum_diff_pipeline,
    pre_since_opened_sum_mean_repeated_pipeline,
    drop_columns_drop_duplicates_pipeline,
)

from classifier import CatBoostEnsembleClassifier

# Logging switch

In [21]:
"""
Переключатель логирования функций пайплайна.
При выборе опции'INFO' будут выводиться названия функций, 
названия исходных обрабатываемых  признаков 
и названия новых фичей.
В классификаторе будут выводиться названия методов,
этапы обучения ансамбля и гиперпараметры моделей ансамбля.
При выборе опции 'OFF' логи отключаются.
"""

log_level_input = input(
    """
Введите уровень логирования для pipeline функций
'INFO' для включения, 'OFF' для отключения
"""
).strip().upper()

if log_level_input == 'OFF':
    # Блокируем логи
    logging.disable(logging.CRITICAL)  
    print("Логирование pipeline функций отключено")

elif log_level_input == 'INFO':
    # Снимаем блокировку
    logging.disable(logging.NOTSET) 

    # Удаляем старые обработчики, чтобы basicConfig сработал
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(message)s'
    )
    print("Логирование pipeline функций включено")

else:
    # Снимаем блокировку
    logging.disable(logging.NOTSET)  

    # Удаляем старые обработчики, чтобы basicConfig сработал
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(message)s'
    )
    print("Неверный ввод, установлен режим INFO")

# Создаём объект логера
logger = logging.getLogger(__name__)


Введите уровень логирования для pipeline функций
'INFO' для включения, 'OFF' для отключения
 off


Логирование pipeline функций отключено


# Load and split data

In [4]:
"""
Собираем исходный датасет из parquet файлов,  
скачиваем только необходимые колонки
"""

# Получаем количество файлов в папке с данными
files_count = check_data_folder_and_count_files(RAW_DATA_PATH, PARQUET_FILE_PATTERN)[1]

# Загружаем датасет
raw_data = load_dataset(
    path_to_dataset=RAW_DATA_PATH,
    num_parts_to_preprocess_at_once=1,
    num_parts_total=files_count,
    save_to_path=TEMP_DATA_PATH,
    verbose=True,
    columns=PRE_FEATURES
)

# Загружаем таргет
# Делим датасет и таргет на train/test
train_test_dict = split_dataset_by_target(
        dataset=raw_data,
        path_to_target=TARGET_PATH,
        train_size=TRAIN_SIZE,
        random_state=SEED_SPLIT_DATASET,
        stratify_col=STRATIFY_COL
    )

2025-08-13 14:47:23,054 - Starting check_data_folder_and_count_files : /Users/admin/DataSince/Machine Learning Junior/21_final_project/21_final_project_5/data/raw
2025-08-13 14:47:23,057 - Count of files in data folder: 12
2025-08-13 14:47:23,058 - Starting load_dataset function
Loading entire data:   0%|                               | 0/12 [00:00<?, ?it/s]2025-08-13 14:47:23,083 - Processing step 0
2025-08-13 14:47:23,084 - Starting load_parquet_chunks function
2025-08-13 14:47:23,086 - Found 12 dataset paths
2025-08-13 14:47:23,087 - Reading chunks:
2025-08-13 14:47:23,088 - /Users/admin/DataSince/Machine Learning Junior/21_final_project/21_final_project_5/data/raw/train_data_0.pq

Reading dataset with pandas: 100%|████████████████| 1/1 [00:00<00:00,  1.61it/s]
2025-08-13 14:47:24,171 - Finished load_parquet_chunks (read 1974724 rows)
2025-08-13 14:47:26,218 - Saved to "/Users/admin/DataSince/Machine Learning Junior/21_final_project/21_final_project_5/data/temp/processed_chunk_000.p

((2400000,), (600000,))

# Pipeline

In [10]:
# Создадим объект классификатора
classifier = CatBoostEnsembleClassifier(
    params_list=PARAMS_LIST,
    weights_list=WEIGHTS_LIST,
    threshold=THRESHOLD,
    cat_features=CAT_FEATURES,
    n_splits=N_SPLITS,
    seed=SEED,
    shuffle=SHUFFLE,
    logger=logger
)
classifier

CatBoostEnsembleClassifier(cat_features=[], logger=<Logger __main__ (INFO)>,
                           params_list=[{'auto_class_weights': 'Balanced',
                                         'bagging_temperature': 0.09710127579,
                                         'boosting_type': 'Plain',
                                         'border_count': 113, 'depth': 4,
                                         'eval_metric': 'AUC',
                                         'grow_policy': 'SymmetricTree',
                                         'iterations': 3000,
                                         'l2_leaf_reg': 8.005778243,
                                         'learning_rate': 0.03625476076,
                                         'loss_function': 'L...
                                         'l2_leaf_reg': 8.005778242558318,
                                         'learning_rate': 0.036254760756236626,
                                         'min_data_in_leaf': 5,
                                         'random_seed': 0,
                                         'random_strength': 8.209932298658357,
                                         'rsm': 0.7343256008238508,
                                         'subsample': 0.9882297325066979,
                                         'verbose': False}],
                           threshold=0.49642,
                           weights_list=[0.7576036850511159, 0.7554545982995526,
                                         0.7532810994057619, 0.7546988571803108,
                                         0.7524269260453276,
                                         0.7546930331964138])

In [14]:
# Соберём пайплайн обработки данных и обучения ансамбля моделей

# Создаём SampleMedianImputer для заполнения пустых значений медианами
imputer = SampleMedianImputer(sample_frac=SAMPLE_FRAC)

# Создаём паплайн препроцессинга данных
preprocessing_pipe = Pipeline([
    ('to_numeric', FunctionTransformer(convert_all_to_numeric_pipeline)),
    ('imputer', imputer),
    ('to_int', FunctionTransformer(convert_all_to_int_pipeline)),
    ('drop_duplicates', FunctionTransformer(drop_duplicates_pipeline))
])

# Создаём основной пайплайн
main_pipe = Pipeline(
    [
        (
            'preprocessing',
            preprocessing_pipe
        ),
        (
            'create_rn_max_feature',
            FunctionTransformer(rn_max_feature_pipeline)
        ),
        (
            'enc_paym_transcoding',
            FunctionTransformer(enc_paym_transcoding_pipeline)
        ),
        (
            'create_definite_value_proportion_features',
            FunctionTransformer(
                partial(definite_value_proportion_features_pipeline, features_dictionary=PROP_FEATURES_DICT)
            )
        ),
        (
            'create_sum_prop_1_feature',
            FunctionTransformer(from_is_zero_prop_1_create_sum_prop_1_feature_pipeline)
        ),
        (
            'create_mean_value_frequency_feature',
            FunctionTransformer(
                partial(mean_value_frequency_feature_pipeline, columns_list=MEAN_FREQ_SOURCE_LIST)
            )
        ),
        (
            'from_enc_paym_create_normalized_group_sum_features_then_diff_features',
            FunctionTransformer(enc_paym_norm_group_sum_diff_pipeline)
        ),
        (
            'from_pre_since_opened_create_pre_since_opened_sum_mean_repeated',
            FunctionTransformer(pre_since_opened_sum_mean_repeated_pipeline)
        ),
        (
            'drop_temporary_and_source_columns_drop_duplicates',
            FunctionTransformer(
                partial(drop_columns_drop_duplicates_pipeline, columns_list=DROP_LIST)
            )
        ),
        (
            'classifier', classifier
        )

    ]
)

In [16]:
# Обучим пайплайн
main_pipe.fit(train_test_dict['X_train'], train_test_dict['y_train'])

2025-08-13 15:06:06,166 - FUNCTION convert_all_to_numeric_pipeline
2025-08-13 15:08:18,769 - FUNCTION convert_all_to_int_pipeline
2025-08-13 15:08:38,174 - FUNCTION drop_duplicates_pipeline
2025-08-13 15:10:32,844 - FUNCTION rn_max_feature_pipeline
2025-08-13 15:10:35,585 - FUNCTION enc_paym_transcoding_pipeline 
2025-08-13 15:10:44,601 - FUNCTION definite_value_proportion_features_pipeline
2025-08-13 15:10:44,602 - Original feature pre_loans_next_pay_summ
2025-08-13 15:10:44,606 - New features
2025-08-13 15:10:44,607 - pre_loans_next_pay_summ_prop_5
2025-08-13 15:10:47,595 - pre_loans_next_pay_summ_prop_0
2025-08-13 15:10:50,209 - Original feature enc_paym_0
2025-08-13 15:10:50,212 - New features
2025-08-13 15:10:50,213 - enc_paym_0_prop_1
2025-08-13 15:10:52,707 - Original feature pre_till_fclose
2025-08-13 15:10:52,709 - New features
2025-08-13 15:10:52,711 - pre_till_fclose_prop_4
2025-08-13 15:10:55,595 - pre_till_fclose_prop_3
2025-08-13 15:10:58,299 - pre_till_fclose_prop_1
2025

Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('to_numeric',
                                  FunctionTransformer(func=<function convert_all_to_numeric_pipeline at 0x14093c180>)),
                                 ('imputer', SampleMedianImputer()),
                                 ('to_int',
                                  FunctionTransformer(func=<function convert_all_to_int_pipeline at 0x14093c220>)),
                                 ('drop_duplicates',
                                  FunctionTransformer(func=<function drop_duplicates_pipeline at 0x14...
                                                          'l2_leaf_reg': 8.005778242558318,
                                                          'learning_rate': 0.036254760756236626,
                                                          'min_data_in_leaf': 5,
                                                          'random_seed': 0,
                                                          'random_strength': 8.209932298658357,
                                                          'rsm': 0.7343256008238508,
                                                          'subsample': 0.9882297325066979,
                                                          'verbose': False}],
                                            threshold=0.49642,
                                            weights_list=[0.7576036850511159,
                                                          0.7554545982995526,
                                                          0.7532810994057619,
                                                          0.7546988571803108,
                                                          0.7524269260453276,
                                                          0.7546930331964138]))])

In [17]:
# Сохраним обученный пайплайн в файл
with open(PIPELINE_PATH, 'wb') as file:
    pickle.dump(main_pipe, file)

In [18]:
# Загрузим обученный пайплайн
with open(PIPELINE_PATH, 'rb') as file:
   main_pipe = pickle.load(file)
main_pipe

Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('to_numeric',
                                  FunctionTransformer(func=<function convert_all_to_numeric_pipeline at 0x14093c180>)),
                                 ('imputer', SampleMedianImputer()),
                                 ('to_int',
                                  FunctionTransformer(func=<function convert_all_to_int_pipeline at 0x14093c220>)),
                                 ('drop_duplicates',
                                  FunctionTransformer(func=<function drop_duplicates_pipeline at 0x14...
                                                          'l2_leaf_reg': 8.005778242558318,
                                                          'learning_rate': 0.036254760756236626,
                                                          'min_data_in_leaf': 5,
                                                          'random_seed': 0,
                                                          'random_strength': 8.209932298658357,
                                                          'rsm': 0.7343256008238508,
                                                          'subsample': 0.9882297325066979,
                                                          'verbose': False}],
                                            threshold=0.49642,
                                            weights_list=[0.7576036850511159,
                                                          0.7554545982995526,
                                                          0.7532810994057619,
                                                          0.7546988571803108,
                                                          0.7524269260453276,
                                                          0.7546930331964138]))])

In [19]:
# Предскажем вероятности классов
pred_proba = main_pipe.predict_proba(train_test_dict['X_test'])
pred_proba

2025-08-13 16:24:59,089 - FUNCTION convert_all_to_numeric_pipeline
2025-08-13 16:25:15,677 - FUNCTION convert_all_to_int_pipeline
2025-08-13 16:25:17,411 - FUNCTION drop_duplicates_pipeline
2025-08-13 16:25:28,435 - FUNCTION rn_max_feature_pipeline
2025-08-13 16:25:28,966 - FUNCTION enc_paym_transcoding_pipeline 
2025-08-13 16:25:30,188 - FUNCTION definite_value_proportion_features_pipeline
2025-08-13 16:25:30,190 - Original feature pre_loans_next_pay_summ
2025-08-13 16:25:30,191 - New features
2025-08-13 16:25:30,194 - pre_loans_next_pay_summ_prop_5
2025-08-13 16:25:30,681 - pre_loans_next_pay_summ_prop_0
2025-08-13 16:25:31,121 - Original feature enc_paym_0
2025-08-13 16:25:31,122 - New features
2025-08-13 16:25:31,123 - enc_paym_0_prop_1
2025-08-13 16:25:31,549 - Original feature pre_till_fclose
2025-08-13 16:25:31,550 - New features
2025-08-13 16:25:31,551 - pre_till_fclose_prop_4
2025-08-13 16:25:32,055 - pre_till_fclose_prop_3
2025-08-13 16:25:32,484 - pre_till_fclose_prop_1
2025

array([[0.94815634, 0.05184366],
       [0.53968526, 0.46031474],
       [0.65102507, 0.34897493],
       ...,
       [0.56323335, 0.43676665],
       [0.83147613, 0.16852387],
       [0.31829477, 0.68170523]])

In [20]:
# Вычислим целевую метрику
roc_auc_score(train_test_dict['y_test'], pred_proba[:,1])

0.7571774366758575

Метрика roc_auc_score получилась практически такой же как и в исследовательском ноутбуке (0.7572337893426191),
думаю разницу можно объяснить погрешностью вычислений, к тому же схемы сбора тренировочных датасетов немного отличается.
В исследовательской части мы набирали признаки  из исходного датасета в датасет с таргетом размером (2400000, 1) а в ноутбуке пайплайна сразу в исходный датасет  размером (20931476, N) и после удаляли дубликаты строк чтобы привести в соответствие с размером таргет датасета.


In [22]:
# Отключим логирование и предскажем класс 1
pred = main_pipe.predict(train_test_dict['X_test'])
pred

array([0, 0, 0, ..., 0, 0, 1])

In [23]:
# Проверим метод трансформера также без логирования
X_test_transformed = main_pipe.transform(train_test_dict['X_test'])
X_test_transformed

,rn_max,pre_loans_next_pay_summ_prop_5,pre_loans_next_pay_summ_prop_0,enc_paym_0_prop_1,pre_till_fclose_prop_4,pre_till_fclose_prop_3,pre_till_fclose_prop_1,enc_loans_credit_type_prop_0,enc_loans_credit_type_prop_2,is_zero_loans5_prop_1,...,pre_loans530_mean_freq,enc_paym_8_mean_freq,pre_loans5_mean_freq,enc_paym_10_mean_freq,enc_loans_account_cur_mean_freq,enc_paym_9_mean_freq,enc_paym_avg_0_1_this_year_diff,enc_paym_avg_1_2_all_diff,enc_paym_avg_0_years_diff,pre_since_opened_repeated_prop
0,10,0.100000,0.100000,0.000000,0.000000,0.000000,0.000000,0.200000,0.1,1.000000,...,0.976465,0.519372,0.994481,0.460134,0.997625,0.492167,11.000000,0.000000,3.000000,0.200000
1,8,0.000000,0.000000,0.125000,0.000000,0.000000,0.000000,0.000000,0.0,0.875000,...,0.976465,0.358095,0.994481,0.484894,0.997625,0.483197,5.625000,1.000000,3.125000,0.250000
2,24,0.000000,0.083333,0.000000,0.041667,0.125000,0.000000,0.041667,0.0,0.958333,...,0.976465,0.497436,0.994481,0.484894,0.997625,0.484131,8.625000,0.250000,5.458333,0.416667
3,10,0.000000,0.100000,0.000000,0.000000,0.100000,0.000000,0.000000,0.3,1.000000,...,0.976465,0.481767,0.994481,0.484894,0.997625,0.483197,7.500000,0.100000,2.600000,0.100000
4,18,0.000000,0.055556,0.055556,0.055556,0.000000,0.000000,0.000000,0.0,0.944444,...,0.976465,0.416355,0.994481,0.498649,0.997625,0.478213,4.833333,0.333333,4.388889,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,6,0.166667,0.166667,0.166667,0.166667,0.000000,0.000000,0.000000,0.0,1.000000,...,0.816428,0.414988,0.994481,0.424269,0.997625,0.483197,5.666667,1.333333,6.000000,0.166667
599996,3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.000000,...,0.976465,0.497436,0.994481,0.474577,0.997625,0.486934,8.000000,0.333333,2.000000,0.000000
599997,5,0.200000,0.000000,0.000000,0.200000,0.000000,0.000000,0.000000,0.0,1.000000,...,0.976465,0.453562,0.994481,0.503464,0.997625,0.476469,5.600000,0.000000,3.000000,0.400000
599998,7,0.285714,0.000000,0.000000,0.285714,0.142857,0.142857,0.142857,0.0,1.000000,...,0.976465,0.475051,0.994481,0.489315,0.997625,0.481595,7.571429,0.000000,3.142857,0.285714


In [24]:
# Сохраним предсказания классов и их вероятностей в файлы

# Создаём имя файла предикта вероятностей
proba_file_name = make_file_path(
    output_type='proba',
    data_path=RAW_DATA_PATH,
    output_dir=TEST_PREDICT_PATH,
    ext=PREDICT_FILE_EXTENSION
)

# Создаём имя файла предикта меток классов
predict_file_name = make_file_path(
    output_type='predict',
    data_path=RAW_DATA_PATH,
    output_dir=TEST_PREDICT_PATH,
    ext=PREDICT_FILE_EXTENSION
)


# Получаем id set для сохранения с предиктом
# Используем drop_duplicates так как X_test это датасет до агрегаций в пайплайне
ids = train_test_dict['X_test']['id'].drop_duplicates().values

# Сохраненяем вероятности в .csv
save_predictions_with_id(
    output_type='proba',
    ids=ids,
    predictions=pred_proba,
    output_path=proba_file_name
)

# Сохраненяем метки классов в .csv
save_predictions_with_id(
    output_type='predict',
    ids=ids,
    predictions=pred,
    output_path=predict_file_name
)